# Go2 ODD Observer - Analysis Demo

**Phase 1.6** - December 2025

This notebook demonstrates the 6-agent pipeline for analyzing robot operational safety.

Pipeline: **OddSpec → Perception + Motion + Collision → Evaluator → Report**

## 1. Setup

Install dependencies and configure the environment.

In [ ]:
# Install dependencies (run once)
import sys
!{sys.executable} -m pip install -q python-dotenv google-genai google-adk matplotlib

print("Dependencies installed")

In [ ]:
import os
import json
from pathlib import Path
from dotenv import load_dotenv

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Load environment variables
load_dotenv(project_root / ".env")

print(f"Project root: {project_root}")

In [ ]:
# Configure Google Gemini API
GOOGLE_API_KEY = os.getenv('GOOGLE_API_KEY')

if GOOGLE_API_KEY:
    print(f"Google Gemini API configured")
else:
    print("GOOGLE_API_KEY not found!")
    print("Get free key: https://aistudio.google.com/app/apikey")

## 2. Understanding ODD and COD

**ODD** (Operational Design Domain): Safe operating envelope for the robot

**COD** (Current Operating Domain): What actually happened during operation

| Category | Examples |
|----------|----------|
| Environment | Lighting, terrain, indoor/outdoor |
| Dynamic | Max speed (2.5 m/s), acceleration |
| Safety | Obstacle density, proximity |

## 3. Available Scenarios

In [ ]:
# List available scenarios
data_dir = project_root / "data"

print("Production Scenarios:")
prod_chunks = data_dir / "production" / "chunks"
if prod_chunks.exists():
    for scenario in sorted(prod_chunks.iterdir()):
        if scenario.is_dir():
            windows = list(scenario.glob("window_*"))
            print(f"   {scenario.name}: {len(windows)} windows")

print("\nTest Scenarios (2-window):")
test_dir = data_dir / "test"
if test_dir.exists():
    for scenario in sorted(test_dir.iterdir()):
        if scenario.is_dir() and not scenario.name.startswith('.'):
            windows = list(scenario.glob("window_*"))
            print(f"   {scenario.name}: {len(windows)} windows")

In [ ]:
# Select a scenario
SCENARIO = "sim_2win"

if SCENARIO.endswith("_2win"):
    scenario_path = data_dir / "test" / SCENARIO
else:
    scenario_path = data_dir / "production" / "chunks" / SCENARIO

if scenario_path.exists():
    windows = sorted(scenario_path.glob("window_*"))
    print(f"Selected: {SCENARIO}")
    print(f"Path: {scenario_path}")
    print(f"Windows: {len(windows)}")
else:
    print(f"Scenario not found: {scenario_path}")

## 4. Explore Window Data

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

window_dir = windows[0]
print(f"Window: {window_dir.name}")
print("\nContents:")
for f in sorted(window_dir.iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f"   {f.name}: {size_kb:.1f} KB")

In [ ]:
# Display camera images
camera_images = sorted(window_dir.glob("camera_*.jpg"))

if camera_images:
    fig, axes = plt.subplots(1, min(3, len(camera_images)), figsize=(15, 5))
    if len(camera_images) == 1:
        axes = [axes]
    
    for ax, img_path in zip(axes, camera_images[:3]):
        img = Image.open(img_path)
        ax.imshow(img)
        ax.set_title(img_path.name)
        ax.axis('off')
    
    plt.suptitle(f"Camera Images - {window_dir.name}")
    plt.tight_layout()
    plt.show()

In [ ]:
# Display BEV channels
bev_channels = ['occupancy', 'height', 'roughness']
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, channel in zip(axes, bev_channels):
    bev_path = window_dir / f"bev_{channel}.png"
    if bev_path.exists():
        img = Image.open(bev_path)
        ax.imshow(img, cmap='gray')
        ax.set_title(f"BEV {channel.title()}")
    ax.axis('off')

plt.suptitle(f"LiDAR BEV - {window_dir.name}")
plt.tight_layout()
plt.show()

print("BEV Semantics:")
print("  Occupancy: Obstacles (ground filtered)")
print("  Height: All points (terrain elevation)")
print("  Roughness: Height variance")

In [ ]:
# Load motion data
motion_file = window_dir / "motion.json"

if motion_file.exists():
    with open(motion_file) as f:
        motion_data = json.load(f)
    
    print(f"Motion frames: {len(motion_data)}")
    speeds = [m.get('derived_speed', 0) for m in motion_data if m.get('derived_speed')]
    if speeds:
        print(f"Speed: {min(speeds):.3f} - {max(speeds):.3f} m/s")

## 5. Run Analysis

In [ ]:
# Run analysis
import subprocess

print(f"Running analysis on: {SCENARIO}")
print("This takes 2-3 minutes...")

result = subprocess.run(
    [sys.executable, "scripts/run_odd_analysis.py", "--scenario", SCENARIO],
    cwd=project_root,
    capture_output=True,
    text=True
)

print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

## 6. Review Results

In [ ]:
# Find latest results
report_dir = data_dir / "report_candidates"

if report_dir.exists():
    timestamps = sorted(report_dir.iterdir(), reverse=True)
    if timestamps:
        latest = timestamps[0]
        print(f"Latest results: {latest.name}")
        for scenario_dir in latest.iterdir():
            if (scenario_dir / "full_result.json").exists():
                print(f"   {scenario_dir.name}")

In [ ]:
# Load results
def load_latest_result(scenario_name):
    report_dir = project_root / "data" / "report_candidates"
    for ts_dir in sorted(report_dir.iterdir(), reverse=True):
        path = ts_dir / scenario_name / "full_result.json"
        if path.exists():
            with open(path) as f:
                return json.load(f), path
    return None, None

result, result_path = load_latest_result(SCENARIO)

if result:
    print(f"Results: {SCENARIO}")
    if 'evaluator' in result and 'output' in result['evaluator']:
        eval_out = result['evaluator']['output']
        print(f"Verdict: {eval_out.get('verdict', 'Unknown')}")
        print(f"Confidence: {eval_out.get('confidence', 0)}%")
    if 'cost' in result:
        print(f"Cost: ${result['cost'].get('total_cost', 0):.4f}")

## 7. Generate HTML Report

In [ ]:
if result_path:
    output_path = project_root / "docs" / "reports" / f"{SCENARIO}_report.html"
    
    gen_result = subprocess.run(
        [sys.executable, "scripts/generate_html_report.py",
         "--input", str(result_path),
         "--scenario-dir", str(scenario_path),
         "--output", str(output_path)],
        cwd=project_root,
        capture_output=True,
        text=True
    )
    
    if gen_result.returncode == 0:
        print(f"Report: {output_path}")
    else:
        print(gen_result.stderr)

## Next Steps

- Try different scenarios: `real_2win`, `real_173442_2win`
- Custom ODD: `--odd "Indoor robot, max 1.5 m/s"`
- Full production runs on 16-window scenarios

See `docs/` for architecture and guides.